In [ ]:
import pandas as pd
import difflib
import re
import matplotlib.pyplot as plt
import os

In [ ]:
import io, json
from ipywidgets import FileUpload, VBox, Button, Output  #type: ignore
from IPython.display import display

uploader = FileUpload(accept='.json', multiple=False)
load_btn = Button(description='Load JSON', button_style='primary')
out = Output()


def json_to_dataframe(json_bytes):
    data = json.loads(json_bytes.decode('utf-8'))
    # Expecting either a list of objects or a dict containing a list of objects
    if isinstance(data, list):
        return pd.DataFrame(data)
    if isinstance(data, dict):
        # try first list-of-dicts value
        for v in data.values():
            if isinstance(v, list) and (len(v) == 0 or isinstance(v[0], dict)):
                return pd.DataFrame(v)
    raise ValueError("Unsupported JSON structure. Provide a list of objects or a dict containing a list of objects.")


df = None  # will be set after clicking "Load JSON"


def on_load_clicked(_):
    global df
    with out:
        out.clear_output()
        if not uploader.value:
            print("Please upload a .json file first.")
            return
        # take first uploaded file
        file_info = next(iter(uploader.value.values()))
        try:
            df = json_to_dataframe(file_info['content'])
            print(f"Loaded {len(df)} rows from JSON.")
        except Exception as e:
            print("Failed to load JSON:", e)


load_btn.on_click(on_load_clicked)
display(VBox([uploader, load_btn, out]))

: 

In [ ]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['year'] = df['date'].dt.year

In [ ]:
def parse_query(query: str):
    years = [int(y) for y in re.findall(r'(19\d{2}|20\d{2})', query)]
    place = None
    # crude extraction: word after 'at' or 'in'
    tokens = query.split()
    if 'at' in tokens:
        idx = tokens.index('at')
        if idx+1 < len(tokens):
            place = tokens[idx+1]
    elif 'in' in tokens:
        idx = tokens.index('in')
        if idx+1 < len(tokens):
            place = tokens[idx+1]
    return {'raw': query, 'place': place, 'years': years}

In [ ]:
def match_place(place: str):
    if not place:
        return None
    candidates = df['district_name'].dropna().unique().tolist() + df['station_name'].dropna().unique().tolist()
    match = difflib.get_close_matches(place, candidates, n=1, cutoff=0.6)
    return match[0] if match else None

In [ ]:
def point_lookup(place: str, year: int):
    m = match_place(place)
    if not m:
        return f"❌ Sorry, I couldn't find a match for '{place}'. Please check the spelling or try a different location."
    
    sub = df[(df['year'] == year) & ((df['district_name'] == m) | (df['station_name'] == m))]
    if sub.empty:
        return f"❌ No water level data available for {m} in {year}. Please try a different year."
    
    row = sub.iloc[0]
    current_level = float(row['currentlevel'])
    level_diff = float(row['level_diff'])
    
    # Determine trend direction
    if level_diff > 0:
        trend = f"📈 increased by {level_diff:.2f}m"
    elif level_diff < 0:
        trend = f"📉 decreased by {abs(level_diff):.2f}m"
    else:
        trend = "➡️ remained stable"
    
    response = f"""🌊 **Water Level Information for {m} in {year}**

📍 **Location Details:**
• Station: {row['station_name']}
• District: {row['district_name']}
• State: {row['state_name']}

📊 **Water Level Data:**
• Current Level: {current_level:.2f} meters
• Change from Previous Year: {trend}

💡 This data represents the groundwater level measurements for the specified location and year."""
    
    return response

In [ ]:
def trend_lookup(place: str, year_from: int, year_to: int):
    m = match_place(place)
    if not m:
        return f"❌ Sorry, I couldn't find a match for '{place}'. Please check the spelling or try a different location."
    
    sub = df[(df['year'] >= year_from) & (df['year'] <= year_to) & ((df['district_name'] == m) | (df['station_name'] == m))]
    if sub.empty:
        return f"❌ No water level data available for {m} between {year_from} and {year_to}. Please try a different year range."
    
    years = sub['year'].tolist()
    values = sub['currentlevel'].tolist()
    
    # Calculate trend statistics
    min_level = min(values)
    max_level = max(values)
    avg_level = sum(values) / len(values)
    total_change = values[-1] - values[0]
    
    # Determine overall trend
    if total_change > 0.5:
        overall_trend = "📈 rising significantly"
    elif total_change > 0:
        overall_trend = "📈 slightly rising"
    elif total_change < -0.5:
        overall_trend = "📉 declining significantly"
    elif total_change < 0:
        overall_trend = "📉 slightly declining"
    else:
        overall_trend = "➡️ relatively stable"
    
    # Create trend chart
    plt.figure(figsize=(8, 5))
    plt.plot(years, values, marker='o', linewidth=2, markersize=6)
    plt.xlabel("Year", fontsize=12)
    plt.ylabel("Water Level (meters)", fontsize=12)
    plt.title(f"Water Level Trend at {m} ({year_from}-{year_to})", fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    fname = f"{m.replace(' ', '_')}_trend_{year_from}_{year_to}.png"
    plt.savefig(fname, dpi=300, bbox_inches='tight')
    plt.close()
    
    response = f"""📈 **Water Level Trend Analysis for {m} ({year_from}-{year_to})**

📍 **Location:** {m}
📅 **Period:** {year_from} to {year_to}

📊 **Trend Summary:**
• Overall Trend: {overall_trend}
• Total Change: {total_change:+.2f} meters
• Average Level: {avg_level:.2f} meters
• Highest Level: {max_level:.2f} meters (in {years[values.index(max_level)]})
• Lowest Level: {min_level:.2f} meters (in {years[values.index(min_level)]})

📈 **Year-by-Year Data:**
{chr(10).join([f"• {year}: {value:.2f}m" for year, value in zip(years, values)])}

📊 **Visualization:** A trend chart has been saved as '{fname}' showing the water level changes over time.

💡 This analysis helps understand groundwater level patterns and can indicate water availability trends in the region."""
    
    return response

In [ ]:
def handle_user(query: str):
    parsed = parse_query(query)
    place, years = parsed['place'], parsed['years']
    
    if not place:
        return "❌ Please specify a location in your query. For example: 'What is the water level in Mumbai in 2020?'"
    
    if len(years) == 1:
        return point_lookup(place, years[0])
    elif len(years) >= 2:
        return trend_lookup(place, years[0], years[1])
    else:
        return "❌ Please specify a year or year range in your query. For example: 'Water level in Mumbai in 2020' or 'Water level trends in Mumbai from 2018 to 2022'"

if __name__ == "__main__":
    print("🌊 Welcome to the Water Level Information Chatbot!")
    print("💡 You can ask questions like:")
    print("   • 'What is the water level in Mumbai in 2020?'")
    print("   • 'Show me water level trends in Delhi from 2018 to 2022'")
    print("   • Type 'exit' or 'quit' to stop\n")
    
    while True:
        q = input("Ask: ")
        if q.lower() in ['exit', 'quit']:
            print("👋 Thank you for using the Water Level Chatbot!")
            break
        ans = handle_user(q)
        print(ans)
        print("\n" + "="*50 + "\n")